<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 2 - Neural Networks</font><br>
<font size=20 color=#00c382>Bank Data</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**

https://archive.ics.uci.edu/dataset/222/bank+marketing

---

# Initialise Notebook

In [1]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [2]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
from matplotlib.pyplot import figure
import matplotlib.ticker as mticker
import matplotlib.animation as animation
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Plotly
import plotly.express as px
import plotly.graph_objects as go

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Pycountry 
import pycountry

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# IPython Display 
from IPython.display import HTML

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [3]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [4]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [5]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [6]:
internet_required = False

In [7]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Load and Explore the Dataset

In [8]:
if online:
    # Fetch the dataset from the Internet
    df = pd.read_csv('bank-full.csv', delimiter=';')
else:
    # Load the dataset from local file
    df = pd.read_csv('bank-full.csv', delimiter=';')

### Explore the dataset

In [9]:
print (df.columns)
print ("")
print (df.shape)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

(45211, 17)


In [10]:
df.sample(20)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
37222,32,management,single,secondary,no,-119,yes,no,cellular,13,may,231,1,370,1,other,no
26221,38,technician,married,secondary,no,1448,yes,no,cellular,20,nov,415,1,-1,0,unknown,no
6569,32,technician,married,secondary,yes,-71,yes,no,unknown,28,may,78,2,-1,0,unknown,no
39915,26,admin.,single,secondary,no,96,no,no,cellular,2,jun,696,1,27,2,failure,no
37247,34,management,single,tertiary,no,76,yes,no,cellular,13,may,747,2,371,3,failure,yes
23302,36,management,married,tertiary,no,0,no,no,telephone,27,aug,645,9,-1,0,unknown,yes
7328,34,admin.,married,secondary,no,1365,yes,no,unknown,29,may,293,2,-1,0,unknown,no
4892,35,blue-collar,married,secondary,no,-557,yes,yes,unknown,21,may,249,1,-1,0,unknown,no
29069,40,retired,married,primary,no,407,yes,no,cellular,2,feb,200,1,271,1,failure,no
35793,39,admin.,married,primary,no,444,yes,no,cellular,8,may,374,2,-1,0,unknown,no


In [11]:
df.dropna(inplace=True)
df.shape

(45211, 17)

This from Gemini - why? Need to understand this and reimplement 

In [12]:

# Convert target variable 'y' to binary (0 and 1)
df['y'] = df['y'].map({'yes': 1, 'no': 0})

# Separate features and target
X = df.drop('y', axis=1)
y = df['y'].values.reshape(-1, 1) # Reshape for matrix multiplication

# One-Hot Encoding for categorical variables
# This converts 'job', 'marital', etc., into multiple numeric columns
X_encoded = pd.get_dummies(X).astype(float)

# Feature Scaling (Min-Max Scaling)
# Neural networks hate large values like 'balance' mixed with small ones like 'day'
X_min = X_encoded.min()
X_max = X_encoded.max()
X_scaled = (X_encoded - X_min) / (X_max - X_min)

# Convert to NumPy array
X_final = X_scaled.to_numpy()

print(f"Original features: {X.shape[1]}")
print(f"Input neurons needed: {X_final.shape[1]}")

Original features: 16
Input neurons needed: 51


---

# Data Cleaning


...do stuff...

### Write out the cleaned dataset for further use



In [13]:
# Create filenames, one timestamped, one 'latest'
filepath = "."
filename = f'{filepath}/bank_full_cleaned_{now_date_timestamp}.csv'
latest_filename = f'{filepath}/bank_full_cleaned_latest.csv'

# Write out CSV files
df.to_csv(filename, index=False)
df.to_csv(latest_filename, index=False)